# N-Body Gravitational Simulation Trainer

This notebook teaches the **N-body problem** — simulating how multiple massive objects move under their mutual gravitational attraction.

## Learning Objectives
- Understand Newton's law of universal gravitation
- See why the N-body problem is computationally hard
- Implement Verlet integration for energy conservation
- Build a binary star system
- Explore the three-body problem

## 1. Newton's Law of Gravitation

Every pair of masses attracts each other with force:

$$\mathbf{F}_{ij} = \frac{G \, m_i \, m_j}{|\mathbf{r}_j - \mathbf{r}_i|^2} \, \hat{\mathbf{r}}_{ij}$$

where $\hat{\mathbf{r}}_{ij}$ points from body $i$ toward body $j$.

The acceleration on body $i$ due to all other bodies:

$$\mathbf{a}_i = \sum_{j \neq i} \frac{G \, m_j \, (\mathbf{r}_j - \mathbf{r}_i)}{|\mathbf{r}_j - \mathbf{r}_i|^3}$$

### Softening

When two bodies get very close, $|\mathbf{r}_{ij}| \to 0$ and the force diverges. We add a **softening parameter** $\varepsilon$:

$$\mathbf{a}_i = \sum_{j \neq i} \frac{G \, m_j \, (\mathbf{r}_j - \mathbf{r}_i)}{(|\mathbf{r}_j - \mathbf{r}_i|^2 + \varepsilon^2)^{3/2}}$$

This prevents numerical singularities without significantly affecting the dynamics at large separations.

## 2. Why the N-Body Problem is Hard

- **2 bodies**: Analytical solution exists (Kepler orbits)
- **3 bodies**: Generally chaotic — no closed-form solution (Poincaré, 1890)
- **N bodies**: $O(N^2)$ pairwise force calculations per timestep

For large N, algorithms like Barnes-Hut ($O(N \log N)$) or Fast Multipole Method ($O(N)$) are needed. We'll stick with direct summation for small N.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Gravitational constant (we'll use G=1 for normalized units)
G = 1.0
softening = 0.01

def compute_accelerations(positions, masses):
    """Compute gravitational accelerations for all bodies."""
    n = len(masses)
    acc = np.zeros_like(positions)
    
    for i in range(n):
        for j in range(i + 1, n):
            r_ij = positions[j] - positions[i]
            dist_sq = np.dot(r_ij, r_ij) + softening**2
            inv_dist_cubed = dist_sq ** (-1.5)
            
            acc[i] += G * masses[j] * inv_dist_cubed * r_ij
            acc[j] -= G * masses[i] * inv_dist_cubed * r_ij
    
    return acc

print("Force computation ready.")

## 3. Verlet Integration

For gravitational systems, we want to conserve energy over long times. The **Velocity Verlet** method is symplectic — it preserves the geometric structure of Hamiltonian mechanics.

The update rules:

$$\mathbf{r}(t + \Delta t) = \mathbf{r}(t) + \mathbf{v}(t)\,\Delta t + \frac{1}{2}\mathbf{a}(t)\,\Delta t^2$$

$$\mathbf{v}(t + \Delta t) = \mathbf{v}(t) + \frac{1}{2}[\mathbf{a}(t) + \mathbf{a}(t + \Delta t)]\,\Delta t$$

Key advantage: energy oscillates around the true value rather than drifting monotonically (as Euler does).

In [ ]:
def verlet_step(positions, velocities, masses, dt):
    """One velocity Verlet step for N-body system."""
    # Current acceleration
    acc = compute_accelerations(positions, masses)
    
    # Update positions
    new_positions = positions + velocities * dt + 0.5 * acc * dt**2
    
    # New acceleration at updated positions
    new_acc = compute_accelerations(new_positions, masses)
    
    # Update velocities
    new_velocities = velocities + 0.5 * (acc + new_acc) * dt
    
    return new_positions, new_velocities

def compute_total_energy(positions, velocities, masses):
    """Compute total energy (kinetic + potential)."""
    n = len(masses)
    
    # Kinetic energy
    KE = 0.0
    for i in range(n):
        KE += 0.5 * masses[i] * np.dot(velocities[i], velocities[i])
    
    # Potential energy
    PE = 0.0
    for i in range(n):
        for j in range(i + 1, n):
            r_ij = positions[j] - positions[i]
            dist = np.sqrt(np.dot(r_ij, r_ij) + softening**2)
            PE -= G * masses[i] * masses[j] / dist
    
    return KE + PE

print("Verlet integrator ready.")

## 4. Binary Star System

Two equal-mass stars orbiting their common center of mass. For a circular orbit of radius $r$ with period $T$:

$$v = \sqrt{\frac{G M}{4r}}$$

where $M$ is the mass of each star and $r$ is the distance from the center.

In [ ]:
# Binary star setup
M = 1e4  # Mass of each star
separation = 2.0  # Distance between stars
r = separation / 2.0  # Distance from center

# Orbital velocity for circular orbit
v_orbit = np.sqrt(G * M / (4 * r))
print(f"Orbital velocity: {v_orbit:.2f}")

# Initial conditions
masses = np.array([M, M])
positions = np.array([
    [-r, 0.0, 0.0],
    [ r, 0.0, 0.0],
], dtype=np.float64)
velocities = np.array([
    [0.0, -v_orbit, 0.0],
    [0.0,  v_orbit, 0.0],
], dtype=np.float64)

# Simulate
dt = 0.0001
n_steps = 20000

trajectory_1 = np.zeros((n_steps, 3))
trajectory_2 = np.zeros((n_steps, 3))
energies = np.zeros(n_steps)

pos, vel = positions.copy(), velocities.copy()
for i in range(n_steps):
    trajectory_1[i] = pos[0]
    trajectory_2[i] = pos[1]
    energies[i] = compute_total_energy(pos, vel, masses)
    pos, vel = verlet_step(pos, vel, masses, dt)

print(f"Simulation complete: {n_steps * dt:.2f} time units")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Orbit plot
axes[0].plot(trajectory_1[:, 0], trajectory_1[:, 1], 'b-', lw=0.5, label='Star 1')
axes[0].plot(trajectory_2[:, 0], trajectory_2[:, 1], 'r-', lw=0.5, label='Star 2')
axes[0].plot(0, 0, 'k+', markersize=10)
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].set_title('Binary Star Orbits')
axes[0].set_aspect('equal')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Energy conservation
t = np.arange(n_steps) * dt
relative_energy_error = (energies - energies[0]) / np.abs(energies[0])
axes[1].plot(t, relative_energy_error, 'k-', lw=0.5)
axes[1].set_xlabel('Time')
axes[1].set_ylabel('ΔE / |E₀|')
axes[1].set_title('Energy Conservation (Verlet)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Max relative energy error: {np.max(np.abs(relative_energy_error)):.2e}")

## 5. Energy Conservation: Euler vs Verlet

Let's compare how well each method preserves energy:

In [ ]:
def euler_nbody_step(positions, velocities, masses, dt):
    """Simple Euler step for N-body."""
    acc = compute_accelerations(positions, masses)
    new_vel = velocities + acc * dt
    new_pos = positions + velocities * dt
    return new_pos, new_vel

# Run both methods
n_compare = 5000
energy_euler = np.zeros(n_compare)
energy_verlet = np.zeros(n_compare)

pos_e, vel_e = positions.copy(), velocities.copy()
pos_v, vel_v = positions.copy(), velocities.copy()

for i in range(n_compare):
    energy_euler[i] = compute_total_energy(pos_e, vel_e, masses)
    energy_verlet[i] = compute_total_energy(pos_v, vel_v, masses)
    pos_e, vel_e = euler_nbody_step(pos_e, vel_e, masses, dt)
    pos_v, vel_v = verlet_step(pos_v, vel_v, masses, dt)

t = np.arange(n_compare) * dt
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t, (energy_euler - energy_euler[0])/np.abs(energy_euler[0]),
        'r-', label='Euler', alpha=0.8)
ax.plot(t, (energy_verlet - energy_verlet[0])/np.abs(energy_verlet[0]),
        'b-', label='Verlet', alpha=0.8)
ax.set_xlabel('Time')
ax.set_ylabel('Relative energy error')
ax.set_title('Energy Conservation: Euler vs Verlet')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 6. Using the `physics_modeling` Package

In [ ]:
from physics_modeling.gravity.nbody import NBodyConfig, NBodySimulation

config = NBodyConfig(G=1.0, softening=0.01, integrator="verlet", dt=0.0001)

sim = NBodySimulation(
    config,
    masses=np.array([1e4, 1e4]),
    positions=np.array([[-1.0, 0.0, 0.0], [1.0, 0.0, 0.0]]),
    velocities=np.array([[0.0, -v_orbit, 0.0], [0.0, v_orbit, 0.0]]),
)

print(f"Number of bodies: {sim.n_bodies}")
print(f"Initial energy: {sim.total_energy:.4f}")

# Run 10000 steps
for _ in range(10000):
    sim.step()

print(f"Final energy: {sim.total_energy:.4f}")
print(f"Positions: {sim.get_positions()}")

## Exercise: Three-Body Problem

Add a third, much lighter body (a planet) to the binary star system.

**Tasks:**
1. Place a body of mass $m = 1.0$ at position $[0, 2, 0]$ with velocity $[v_p, 0, 0]$
2. Choose $v_p$ so the planet orbits the center of mass (hint: $v_p \approx \sqrt{2GM/r}$)
3. Simulate for a long time and plot the trajectory
4. Try slightly different initial conditions — observe chaos!
5. Track energy conservation — does it still work well with 3 bodies?

In [ ]:
# Your three-body solution here
m_planet = 1.0
r_planet = 2.0
v_planet = np.sqrt(2 * G * M / r_planet)  # Approximate orbital velocity

masses_3 = np.array([M, M, m_planet])
positions_3 = np.array([
    [-r, 0.0, 0.0],
    [ r, 0.0, 0.0],
    [0.0, r_planet, 0.0],
])
velocities_3 = np.array([
    [0.0, -v_orbit, 0.0],
    [0.0,  v_orbit, 0.0],
    [v_planet, 0.0, 0.0],
])

# TODO: Simulate with NBodySimulation and plot the planet's trajectory
# Try perturbing the planet's initial position by 0.01 and compare